<a href="https://colab.research.google.com/github/PalakPrajapati346/WORKSHOP-2/blob/main/Copy_of_flipkart_girdlock.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

In [13]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [14]:
def advanced_pipeline(df, is_train=True, target_map=None):
    df['timestamp'] = pd.to_datetime(df['timestamp'], format='%H:%M')

    # Precision Time Features
    df['day_minutes'] = df['timestamp'].dt.hour * 60 + df['timestamp'].dt.minute
    df['time_sin'] = np.sin(2 * np.pi * df['day_minutes'] / 1440)
    df['time_cos'] = np.cos(2 * np.pi * df['day_minutes'] / 1440)

    # Lane Capacity
    df['lane_capacity'] = df['NumberofLanes'] * (df['RoadType'].map({'Highway':3, 'Urban':2, 'Rural':1}).fillna(1))

    # Target Encoding
    if is_train:
        target_map = train.groupby('geohash')['demand'].mean().to_dict()
    df['geohash_avg_demand'] = df['geohash'].map(target_map).fillna(train['demand'].mean())

    le = LabelEncoder()
    for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        df[col] = le.fit_transform(df[col].astype(str))

    cols_to_drop = ['timestamp', 'geohash', 'Index', 'demand', 'day_minutes']
    return df.drop([c for c in cols_to_drop if c in df.columns], axis=1), target_map

In [15]:
X_all, t_map = advanced_pipeline(train, is_train=True)
y_all = train['demand']
X_test_final, _ = advanced_pipeline(test, is_train=False, target_map=t_map)
X_test_final = X_test_final[X_all.columns]

In [16]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_all)) # Out-of-fold predictions for local score
test_preds = np.zeros(len(X_test_final))

In [17]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.01, # Slow and precise
    'num_leaves': 255,     # High capacity
    'feature_fraction': 0.7,
    'n_estimators': 5000,
    'importance_type': 'gain',
    'verbosity': -1,
    'random_state': 42
}

In [18]:
for fold, (train_idx, val_idx) in enumerate(kf.split(X_all, y_all)):
    X_train, X_val = X_all.iloc[train_idx], X_all.iloc[val_idx]
    y_train, y_val = y_all.iloc[train_idx], y_all.iloc[val_idx]

    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=0)])

    # Store validation predictions
    oof_preds[val_idx] = model.predict(X_val)
    # Average test predictions across folds
    test_preds += model.predict(X_test_final) / 5
    print(f"Fold {fold+1} finished.")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[722]	valid_0's rmse: 0.0373864
Fold 1 finished.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[674]	valid_0's rmse: 0.037428
Fold 2 finished.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[731]	valid_0's rmse: 0.0372758
Fold 3 finished.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[851]	valid_0's rmse: 0.0375482
Fold 4 finished.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[860]	valid_0's rmse: 0.0379927
Fold 5 finished.


In [19]:
r2 = r2_score(y_all, oof_preds)
print(f"\nFINAL PROJECTED HACKATHON SCORE: {max(0, 100 * r2):.4f}")


FINAL PROJECTED HACKATHON SCORE: 93.0345


In [20]:
# 5. Save Submission
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission.to_csv('submission.csv', index=False)